# Transformers: Self-Attention
This is based on an assignment authored by Kabir Ahuja.


In [ ]:
%%bash

pip install torch
pip install datasets
mkdir -p data/embeddings/glove.6B
wget https://homes.cs.washington.edu/~minjang/cse446/glove.6B.50d.txt -O data/embeddings/glove.6B/glove.6B.50d.txt
mkdir -p data
wget https://homes.cs.washington.edu/~minjang/cse446/shakespear_train.txt -O data/shakespear_train.txt
wget https://homes.cs.washington.edu/~minjang/cse446/shakespear_dev.txt -O data/shakespear_dev.txt
wget https://homes.cs.washington.edu/~minjang/cse446/shakespear_test.txt -O data/shakespear_test.txt

Note: **Make sure to use a T4 runtime. With the GPU, training took about 20 minutes.**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gc
import re
import numpy as np
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
###################################
#### GIVEN CODE. DO NOT MODIFY ####
###################################

# Word Embeddings
class GloveEmbeddings:
    def __init__(self, path="data/embeddings/glove.6B/glove.6B.50d.txt"):
        self.path = path
        self.vec_size = int(re.search(r"\d+(?=d)", path).group(0))
        self.embeddings = {}
        self.load()
        self.vocab = list(self.embeddings.keys())
        self.word2idx = {word: i for i, word in enumerate(self.vocab)}

    def __len__(self):
        return len(self.embeddings)

    def load(self):
        for line in open(self.path, "r"):
            values = line.split()

            word_len = len(values) - self.vec_size

            word = " ".join(values[:word_len])
            vector_values = list(map(float, values[word_len:]))

            word = values[0]
            vector_values = list(map(float, values[-self.vec_size:]))
            vector = torch.tensor(vector_values, dtype=torch.float)
            self.embeddings[word] = vector

    def is_word_in_embeddings(self, word):
        return word in self.embeddings

    def get_vector(self, word):
        if not self.is_word_in_embeddings(word):
            return self.embeddings["unk"]
        return self.embeddings[word]

    # Use square operator to get the vector of a word
    def __getitem__(self, word):
        return self.get_vector(word)


### Data Preprocessing

In [ ]:
###################################
#### GIVEN CODE. DO NOT MODIFY ####
###################################

with open("data/shakespear_train.txt", "r") as f: lines_train = f.readlines()
with open("data/shakespear_dev.txt", "r") as f: lines_dev = f.readlines()
with open("data/shakespear_test.txt", "r") as f: lines_test = f.readlines()

# each element is a list of tokens
tokens_train = [line.split() for line in lines_train]

print(f"train docs: {len(tokens_train)}")
print(f"total train tokens: {sum(len(t) for t in tokens_train)}")


# utility fn to flatten the tokens structure
def flat(tokens):
    for t in tokens:
        yield from t

# get counts of each token sorted by count, descending
# also add a few special tokens (with high counts) so they appear first
token_counts = Counter(flat(tokens_train))
token_counts["<START>"] = 1000004
token_counts["<STOP>"] = 1000003
token_counts["<UNK>"] = 1000002
token_counts["<PAD>"] = 1000001
sorted_tokens = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)

print("unique_tokens:", len(token_counts))
print("unique_tokens, count>=3:", len([t for t in sorted_tokens if t[1] >= 3]))

# make tokenizer for all tokens with count >= 3
# note that our tokenizer ends up including START and STOP tokens too
tokenizer = {t[0]: i for i, t in enumerate(sorted_tokens) if t[1] >= 3}


def pad_to_length(tokens, max_len, tokenizer=tokenizer):
    return tokens[:max_len] + [tokenizer["<PAD>"]] * (max_len - len(tokens))


def tokenize(sentence, pad_to_len=None, include_stop=True, tokenizer=tokenizer):
    words = [tokenizer.get(w, tokenizer["<UNK>"]) for w in sentence.split()]
    # add START and STOP tokens
    tokens = [tokenizer["<START>"]] + words + ([tokenizer["<STOP>"]] * include_stop)

    if pad_to_len is not None:
        tokens = pad_to_length(tokens, pad_to_len, tokenizer=tokenizer)
    return tokens


# invert tokenizer for decoding
tokenizer_inv = {v: k for k, v in tokenizer.items()}


def decode(tokens, tokenizer_inv=tokenizer_inv, end_at_stop=True, omit_pad=True):
    tokens = [tokenizer_inv[t] for t in tokens]
    if omit_pad:
        tokens = [t for t in tokens if t != "<PAD>"]
    if end_at_stop and "<STOP>" in tokens:
        tokens = tokens[: tokens.index("<STOP>") + 1]
    return " ".join(tokens)

### Below is an example of tokenization, and how to encode and decode them.

In [ ]:
sentence = "More people have said an Escher sentence than I have ."
tokenized = tokenize(sentence, pad_to_len=25)  # pad to only 25 so it looks nice
decoded = decode(tokenized, end_at_stop=False, omit_pad=False)
print(f"{sentence=}\n{tokenized=}\n{decoded=}")

In [ ]:
MAX_LEN = 100

data_train = torch.tensor(
    [tokenize(t, MAX_LEN) for t in lines_train if len(t) > 0], dtype=torch.long
)
data_val = torch.tensor(
    [tokenize(t, MAX_LEN) for t in lines_dev if len(t) > 0], dtype=torch.long
)

In [ ]:
###################################
#### GIVEN CODE. DO NOT MODIFY ####
###################################

glove = GloveEmbeddings()

# X is all but last token, Y is all but first token
train_dataset = torch.utils.data.TensorDataset(data_train[:, :-1], data_train[:, 1:])
val_dataset = torch.utils.data.TensorDataset(data_val[:, :-1], data_val[:, 1:])
vocab_size = len(glove)

# Define tokenizer_vocab_size
tokenizer_vocab_size = len(tokenizer)

# Create embedding_matrix from glove and tokenizer_inv
embedding_matrix_list = []
for token_idx in range(tokenizer_vocab_size):
    word = tokenizer_inv[token_idx]
    # glove[word] implicitly handles <UNK> if the word is not in glove's vocab
    embedding_matrix_list.append(glove[word])
embedding_matrix = torch.stack(embedding_matrix_list).to(device)


# Implementing the Transformer


In this section, you will implement self attention:
$$\text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^\top}{\sqrt{d_k}})V$$
In addition, you will also implement multi-headed attention (MHA).

When moving from a single head of attention to MHA, each head of attention ($Q_h, K_h, V_h$ matrices) is $\text{d_head} \times \text{d_model}$, and the overall $Q, K, V$ matrices are simply each $Q_h, K_h, V_h$ matrix stacked on top of each other. This means that the overall $Q, K, V$ matrices are $\text{d_model} \times \text{d_model}$. This adds an additional requirement that $\text{d_model}$ must be divisible by the number of heads.


In [ ]:
class LM(nn.Module):
  """
  Single Layer Transformer Block with Multi-Head Attention
  """
  def __init__(self, d_model, num_heads, tokenizer_vocab_size, embedding_matrix):
    super(LM, self).__init__()
    self.d_model = d_model
    self.num_heads = num_heads

    # Ensure d_model is perfectly divisible by the number of heads
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
    self.d_k = d_model // num_heads

    # Add an embedding layer that uses the pre-trained GloVe embeddings
    self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True)

    # Note: These project to d_model, NOT d_k. We will split them into heads later.
    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)

    # Final linear projection after concatenating the heads
    self.W_o = nn.Linear(d_model, d_model)

    self.MLP = nn.Sequential(
        nn.Linear(d_model, 4*d_model),
        nn.ReLU(),
        nn.Linear(4*d_model, 4*d_model),
        nn.ReLU(),
        nn.Linear(4*d_model, d_model)
    )

    self.LM_head = nn.Linear(d_model, tokenizer_vocab_size)

  def apply_rope(self, x):
    """
    RoPE positional encoding. This is already implemented for you.
    """
    # x: (B, H, L, d_k)
    L, D = x.shape[-2], x.shape[-1]
    half = D // 2
    inv_freq = 1.0 / (10000 ** (torch.arange(half, device=x.device).float() / half))
    angles = torch.arange(L, device=x.device).float()[:, None] * inv_freq[None, :]
    cos, sin = angles.cos(), angles.sin()
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

  def self_attention(self, Q: torch.tensor, K: torch.tensor, V: torch.tensor) -> torch.tensor:
    """
    Compute the causally-masked scaled dot-product self-attention.
    Q, K, V shapes: (B, H, L, d_k)
    Returns: (B, H, L, d_k)
    """
    # Step 1: Compute attention scores -> (B, H, L, L)
    scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)

    # Step 2: Apply causal mask (upper triangular positions are not allowed to attend)
    L = scores.size(-1)
    causal_mask = torch.triu(
        torch.ones(L, L, device=scores.device, dtype=torch.bool),
        diagonal=1,
    )
    scores = scores.masked_fill(causal_mask, float("-inf"))

    # Step 3: Softmax along the key axis and multiply by V
    attn = torch.softmax(scores, dim=-1)
    return attn @ V  # (B, H, L, d_k)

  def multi_head_attention(self, input_embeddings: torch.tensor) -> torch.tensor:
    """
    Handles projections, splitting into heads, calling self_attention, and concatenating.
    input_embeddings: (B, L, d_model)
    Returns: (B, L, d_model)
    """
    B, L, _ = input_embeddings.size()

    # Step 1: Q, K, V projections -> (B, L, d_model)
    Q = self.W_q(input_embeddings)
    K = self.W_k(input_embeddings)
    V = self.W_v(input_embeddings)

    # Step 2: Reshape into heads -> (B, H, L, d_k)
    Q = Q.view(B, L, self.num_heads, self.d_k).transpose(1, 2)
    K = K.view(B, L, self.num_heads, self.d_k).transpose(1, 2)
    V = V.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

    # Step 3: Apply RoPE positional encoding to Q and K
    Q = self.apply_rope(Q)
    K = self.apply_rope(K)

    # Step 4: Self-attention, then concatenate heads back to (B, L, d_model)
    attn_out = self.self_attention(Q, K, V)                       # (B, H, L, d_k)
    attn_out = attn_out.transpose(1, 2).contiguous()              # (B, L, H, d_k)
    attn_out = attn_out.view(B, L, self.d_model)                  # (B, L, d_model)

    # Final output projection
    return self.W_o(attn_out)

  def forward(self, input_ids: torch.tensor) -> torch.tensor:
    """
    Compute the forward pass of the model. The output should be of shape (B, L, vocab_size).
    """
    # Step 1: Convert input_ids to embeddings -> (B, L, d_model)
    x = self.embedding(input_ids)

    # Step 2: Transformer block with residual connections (MHA, then MLP)
    x = x + self.multi_head_attention(x)
    x = x + self.MLP(x)

    # Step 3: Logits over the tokenizer vocabulary -> (B, L, vocab_size)
    return self.LM_head(x)


In [ ]:
model = LM(50, 5, tokenizer_vocab_size, embedding_matrix)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

# Step 2: Train your LM
#### This has been implemented for you


In [ ]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    loss_fn: nn,
    optimizer: optim,
    device: str,
    num_epochs,
    tokenizer_vocab_size # Pass tokenizer_vocab_size for loss calculation
):
  train_losses = []
  val_losses = []
  model.train()
  model.to(device)
  for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
      optimizer.zero_grad()
      inputs, targets = batch # inputs: (B, L), targets: (B, L)
      inputs = inputs.to(device)
      targets = targets.to(device) # Move targets to device as well

      outputs = model(inputs) # outputs: (B, L, tokenizer_vocab_size)

      # Reshape outputs and targets for CrossEntropyLoss
      # outputs needs to be (N, C) -> (B * L, tokenizer_vocab_size)
      # targets needs to be (N) -> (B * L)
      outputs = outputs.view(-1, tokenizer_vocab_size)
      targets = targets.view(-1)

      # CrossEntropyLoss expects class indices for targets, not one-hot encoding
      loss = loss_fn(outputs, targets)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_loss = evaluate(model, val_loader, loss_fn, device)
    val_losses.append(val_loss)
  return train_losses, val_losses

def evaluate(model: nn.Module, val_loader: DataLoader, loss_fn: nn, device: str):
  model.eval()
  total_loss = 0.0
  with torch.no_grad():
    for batch in val_loader:
      inputs, targets = batch
      inputs = inputs.to(device)
      targets = targets.to(device)
      outputs = model(inputs)
      outputs = outputs.view(-1, tokenizer_vocab_size)
      targets = targets.view(-1)
      loss = loss_fn(outputs, targets)
      total_loss += loss.item()
  return total_loss / len(val_loader)

# Re-initialize the model with the corrected parameters
model = LM(50, 5, tokenizer_vocab_size, embedding_matrix) # Pass correct arguments
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, shuffle=False)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)
num_epochs = 150
# Pass tokenizer_vocab_size to the train function
train_losses, val_losses = train(model, train_loader, val_loader, loss_fn, optimizer, device, num_epochs, tokenizer_vocab_size)

plt.figure(figsize=(12, 6))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over Epochs")
plt.legend()
plt.show()

# Try it out!
#### Test your LM by generating some text

In [ ]:
def generate(starting_text, max_generation_tokens=20):
  initial_word_list = starting_text.split()
  current_token_ids_list = [tokenizer["<START>"]] + \
                            [tokenizer.get(word, tokenizer["<UNK>"]) for word in initial_word_list]

  running_token_ids = torch.tensor([current_token_ids_list], dtype=torch.long, device=device)
  generated_words = [tokenizer_inv[idx] for idx in current_token_ids_list]

  for _ in range(max_generation_tokens):
    input_seq_len = running_token_ids.size(1)
    if input_seq_len > MAX_LEN - 1:
        input_for_model = running_token_ids[:, -(MAX_LEN - 1):]
    else:
        input_for_model = running_token_ids

    outputs = model(input_for_model)
    logits = outputs[0, -1, :].cpu().detach().numpy() # Logits for the next token
    probabilities = np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))
    next_token_idx = np.random.choice(np.arange(len(probabilities)), p=probabilities)
    next_word = tokenizer_inv[next_token_idx]
    generated_words.append(next_word)

    next_token_tensor = torch.tensor([[next_token_idx]], dtype=torch.long, device=device)
    running_token_ids = torch.cat([running_token_ids, next_token_tensor], dim=1)

    if next_word == "<STOP>":
      break

  final_text_words = []
  for word in generated_words:
      if word not in ["<START>", "<PAD>"]:
          final_text_words.append(word)
      if word == "<STOP>":
          break

  if not final_text_words:
      return "<empty>"
  return " ".join(final_text_words)

print("--- Generation for '<START>' ---")
print(generate("<START>"))
print("\n--- Generation for 'KING RICHARD' ---")
print(generate("KING RICHARD II: "))
print("\n--- Generation for 'To be or not?' ---")
print(generate("To be or not "))
